<img src="https://www.funcionpublica.gov.co/documents/d/guest/logo-universidad-nacional" alt="Logo UNAL" width="600"/>

### **Universidad Nacional de Colombia sede Manizales**
#### Facultad de ingeniería y arquitectura
#### Departamento de ingeniería eléctrica, electrónica y computación
#### *Procesamiento del Lenguaje Natural*

#### Profesor: Lucas Iturriago
#### Monitora: Isabella Valero Mora - lvalerom@unal.edu.co

## 1. Modelos Secuenciales
En las clases anteriores (BoW, TF-IDF), tratábamos el texto como un "saco de palabras" donde el orden no importaba. Sin embargo, en el lenguaje humano, la posición de una palabra cambia radicalmente el sentido:
* **Frase A:** "No, él es un spammer."
* **Frase B:** "Él no es un spammer."

Los **Modelos Secuenciales** (como las RNN y LSTMs) procesan la información palabra por palabra, manteniendo una "memoria" de lo que ocurrió antes. 

### Pytorch el motor de aprendizaje profundo
**PyTorch** nos obliga a definir explícitamente cómo fluyen los datos. Esto es más laborioso, pero nos da un control total sobre el gradiente y la arquitectura.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from collections import Counter
import re

# Configuración de hardware
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo detectado: {device}")

### 2. Carga y Exploración de Datos
Antes de modelar, debemos entender a qué nos enfrentamos. El dataset *SMS Spam Collection* es un clásico, pero presenta un desafío común en NLP: el **desbalance de clases**.

In [ ]:
# Carga del dataset desde la URL corregida
url = "https://raw.githubusercontent.com/UN-GCPDS/ProcesamientoLenguajeNatural/refs/heads/main/Insumos/SMSSpamCollection.txt"
df = pd.read_csv(url, sep='\t', header=None, names=['label', 'message'])

# Visualización de la distribución de las categorías
print("Distribución de clases:")
print(df['label'].value_counts())

### 3. Preprocesamiento: De Texto a Tensores
Las redes neuronales solo entienden de números (tensores). El proceso requiere tres pasos críticos:
1. **Codificación de etiquetas:** Convertir 'ham'/'spam' en 0 y 1.
2. **Tokenización:** Dividir las oraciones en unidades mínimas (palabras).
3. **Padding (Acolchado):** Como las oraciones tienen longitudes distintas, debemos nivelarlas para que la GPU pueda procesarlas en bloques (batches).



In [ ]:
# 1. Codificación de etiquetas: ham -> 0, spam -> 1
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

# 2. Tokenización: Limpieza y división de palabras
def tokenize(text):
    # Pasamos a minúsculas y quitamos caracteres especiales
    text = re.sub(r'[^\w\s]', '', text.lower())
    return text.split()

# 3. Construcción del Vocabulario Saneado
all_words = []
for msg in df['message']:
    all_words.extend(tokenize(msg))

counts = Counter(all_words)

# Extraemos las palabras que cumplen la condición
frequent_words = [word for word, count in counts.items() if count > 1]

# Enumeramos solo las palabras filtradas
# Reservamos 0 para <PAD> y 1 para <UNK>
vocab = {word: i+2 for i, word in enumerate(frequent_words)}
vocab["<PAD>"] = 0 
vocab["<UNK>"] = 1

vocab_size = len(vocab)
print(f"Tamaño real del vocabulario: {vocab_size}")
# Verificación: el índice máximo debe ser vocab_size - 1
print(f"Índice máximo en vocab: {max(vocab.values())}")

### La importancia del Padding
Para no desperdiciar memoria, no usaremos la longitud máxima absoluta, sino el **percentil 95**. Esto significa que cubriremos el 95% de los mensajes sin recortarlos demasiado.

In [ ]:
# Calcular longitud máxima basada en el percentil 95
df['msg_len'] = df['message'].apply(lambda x: len(tokenize(x)))
max_length = int(df['msg_len'].quantile(0.95))
print(f"Longitud de secuencia seleccionada (95% de los datos): {max_length}")

def encode_and_pad(text, vocab, max_len):
    """Convierte texto en una lista de índices de longitud fija."""
    tokens = tokenize(text)
    # Buscamos en el vocabulario, si no existe usamos el índice 1 (<UNK>)
    encoded = [vocab.get(w, 1) for w in tokens]
    
    # Aplicamos padding (relleno con 0) o truncamos según la longitud máxima
    if len(encoded) < max_len:
        encoded += [0] * (max_len - len(encoded))
    else:
        encoded = encoded[:max_len]
    return encoded

# Transformar el dataset completo a matrices numéricas
X = np.array([encode_and_pad(m, vocab, max_length) for m in df['message']])
y = df['label'].values

# Dividir en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Forma de X_train: {X_train.shape}")

### 4. El Objeto Dataset y DataLoader de PyTorch
En PyTorch, los datos se encapsulan en una clase `Dataset`. Esto permite que el `DataLoader` gestione el barajado de datos y el tamaño de los lotes de forma eficiente.

In [ ]:
class SpamDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)
        
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Creación de los cargadores de datos
train_loader = DataLoader(SpamDataset(X_train, y_train), batch_size=32, shuffle=True)
test_loader = DataLoader(SpamDataset(X_test, y_test), batch_size=32)

# Verificamos la forma del primer batch
sample_x, sample_y = next(iter(train_loader))
print(f"Forma del lote (Batch Size, Seq Length): {sample_x.shape}")

## 1. Anatomía de una RNN (SimpleRNN)
Una Red Neuronal Recurrente no procesa todo el mensaje de un solo golpe. Lo hace palabra por palabra (o paso a paso en el tiempo). 

### El Estado Oculto (Hidden State)
La clave de la RNN es el **Hidden State ($h_t$)**. Imagina que es una pequeña libreta de notas donde la red escribe un resumen de lo que ha leído hasta ahora. Al recibir la siguiente palabra, la red lee su libreta ($h_{t-1}$), mira la nueva palabra ($x_t$) y actualiza su libreta ($h_t$).



### Capas del Modelo:
1. **Embedding:** Convierte los índices de palabras en vectores densos de significado.
2. **RNN Layer:** La "memoria" que procesa la secuencia.
3. **Dropout:** Una técnica de regularización que "apaga" neuronas al azar para evitar que el modelo se memorice los datos (overfitting).
4. **Fully Connected (Dense):** La capa final que decide si el resumen acumulado indica 'spam' o 'ham'.

In [ ]:
class SimpleRNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(SimpleRNNModel, self).__init__()
        
        # 1. Capa de inmersión: transforma enteros en vectores semánticos
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # 2. Capa RNN: 'batch_first=True' indica que nuestros datos vienen como (Batch, Seq, Feature)
        self.rnn = nn.RNN(embedding_dim, hidden_dim, batch_first=True)
        
        # 3. Regularización
        self.dropout = nn.Dropout(0.5)
        
        # 4. Capa de salida: de la dimensión oculta a una sola neurona de clasificación
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        # x: [Batch Size, Seq Length]
        embedded = self.embedding(x) # [Batch Size, Seq Length, Embedding Dim]
        
        # out: contiene todos los hidden states de la secuencia
        # hidden: contiene solo el último estado oculto (el 'resumen' final)
        out, hidden = self.rnn(embedded)
        
        # Usamos el último estado oculto para la clasificación final
        # hidden tiene forma [1, Batch Size, Hidden Dim] -> necesitamos quitar la dimensión 1
        final_state = self.dropout(hidden.squeeze(0))
        
        return self.fc(final_state)

# Instanciamos el modelo
# Usaremos 100 dimensiones para los vectores de palabras y 128 para la memoria interna
EMBEDDING_DIM = 100
HIDDEN_DIM = 128
model = SimpleRNNModel(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, 1).to(device)

print(model)

### 2. Configuración del Entrenamiento: Pérdida y Optimización
En PyTorch, debemos definir explícitamente cómo vamos a medir el error y cómo vamos a actualizar los pesos.

* **Función de Pérdida:** Usaremos `BCEWithLogitsLoss`. Esta función combina una capa Sigmoide con la Entropía Cruzada Binaria, lo cual es numéricamente más estable para problemas de clasificación de dos clases.
* **Optimizador:** Usaremos **Adam**, un algoritmo que ajusta automáticamente la tasa de aprendizaje para cada parámetro del modelo.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

### 3. El Bucle de Entrenamiento (Training Loop)
A diferencia de `model.fit()` en Keras, en PyTorch debemos programar el ciclo manualmente. Esto nos permite un control total sobre el proceso.

El flujo estándar es:
1. **Zero Grad:** Limpiar los gradientes del paso anterior.
2. **Forward:** Pasar los datos por el modelo.
3. **Loss:** Calcular qué tan lejos estamos de la respuesta correcta.
4. **Backward:** Calcular cuánto debe cambiar cada peso (backpropagation).
5. **Step:** Actualizar los pesos.

In [ ]:
def train_model(model, train_loader, optimizer, criterion, epochs=5):
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        correct = 0
        total = 0
        
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            # 1. Reiniciar gradientes
            optimizer.zero_grad()
            
            # 2. Forward pass
            predictions = model(batch_x).squeeze(1) # Ajustar forma a [Batch Size]
            
            # 3. Calcular pérdida
            loss = criterion(predictions, batch_y)
            
            # 4. Backpropagation
            loss.backward()
            
            # 5. Actualizar pesos
            optimizer.step()
            
            epoch_loss += loss.item()
            
            # Métricas rápidas: convertimos logits a predicciones binarias
            preds = torch.round(torch.sigmoid(predictions))
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
            
        print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f} | Acc: {correct/total:.4f}")

# Entrenar el modelo
train_model(model, train_loader, optimizer, criterion)

## 1. LSTMs y GRUs: Memoria con Esteroides
Como vimos, las RNN simples olvidan rápido. Para solucionar esto, surgieron las **LSTM (Long Short-Term Memory)** y las **GRU (Gated Recurrent Units)**.

### ¿Cómo funcionan las "Puertas"?
A diferencia de la RNN, una LSTM tiene un **Estado de Celda (Cell State)**, que es como una cinta transportadora de información que atraviesa toda la secuencia. La red utiliza "puertas" lógicas para gestionar esta cinta:
* **Puerta de Olvido (Forget Gate):** Decide qué información vieja ya no es útil y debe borrarse.
* **Puerta de Entrada (Input Gate):** Decide qué información nueva merece ser guardada.
* **Puerta de Salida (Output Gate):** Decide qué parte de la memoria se mostrará en el paso actual.



La **GRU** es una versión simplificada de la LSTM que combina estas puertas, siendo más rápida de entrenar y casi igual de precisa.

In [ ]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Capa LSTM: devuelve el 'output' y una tupla (hidden_state, cell_state)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, batch_first=True)
        
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, output_dim)
        
    def forward(self, x):
        embedded = self.embedding(x)
        
        # La LSTM devuelve:
        # out: [Batch, Seq, Hidden] (todos los estados)
        # (h_n, c_n): Los últimos estados ocultos y de celda
        out, (hidden, cell) = self.lstm(embedded)
        
        # Extraemos el último hidden state [Batch, Hidden]
        # h_n tiene forma [num_layers, Batch, Hidden]
        final_state = self.dropout(hidden[-1])
        
        return self.fc(final_state)

# Instanciar el modelo LSTM
lstm_model = LSTMModel(vocab_size, EMBEDDING_DIM, HIDDEN_DIM, 1).to(device)
optimizer_lstm = optim.Adam(lstm_model.parameters(), lr=0.001)

### 2. Entrenamiento y Evaluación
Entrenaremos nuestra LSTM y luego evaluaremos su capacidad de generalización con datos que nunca ha visto (el Test Set).

In [ ]:
print("Entrenando Modelo LSTM...")
train_model(lstm_model, train_loader, optimizer_lstm, criterion, epochs=5)

def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad(): # Desactivamos el cálculo de gradientes para ahorrar memoria
        for batch_x, batch_y in loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x).squeeze(1)
            preds = torch.round(torch.sigmoid(outputs))
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
            
    print(f"\nExactitud en el conjunto de prueba: {correct/total:.4f}")

evaluate_model(lstm_model, test_loader)

### 3. Inferencia: Probando con tus propios mensajes
Un modelo solo es útil si puede predecir sobre datos nuevos. Crearemos una función que tome un texto crudo, lo transforme al formato del vocabulario y nos diga si es Spam o Ham.

In [ ]:
def predict_sms(message, model, vocab, max_len):
    model.eval()
    # 1. Preprocesar y codificar
    encoded = encode_and_pad(message, vocab, max_len)
    tensor_input = torch.tensor([encoded], dtype=torch.long).to(device)
    
    # 2. Predecir
    with torch.no_grad():
        output = model(tensor_input).squeeze(1)
        prob = torch.sigmoid(output).item()
        
    result = "SPAM" if prob > 0.5 else "HAM"
    print(f"Mensaje: {message}")
    print(f"Probabilidad de Spam: {prob:.4f} -> Resultado: {result}\n")

# Pruebas en "vivo"
predict_sms("WINNER! You have won a 1000 cash prize. Text 'CLAIM' to 8888", lstm_model, vocab, max_length)
predict_sms("Hey mom, I will be home for dinner at 7pm.", lstm_model, vocab, max_length)

## Ejercicio 

**Contexto:** Has visto cómo la RNN y la LSTM manejan la memoria. Existe una tercera variante llamada **GRU (Gated Recurrent Unit)** que suele ser más eficiente en datasets pequeños como este.

**Tu reto:**
1. Crea una nueva clase llamada `GRUModel` utilizando `nn.GRU`. Ten en cuenta que la GRU no tiene `cell_state`, por lo que solo devuelve `out, hidden`.
2. Entrena el modelo durante 5 épocas.
3. Compara el tiempo de entrenamiento y la exactitud final frente a la LSTM.
4. **Pregunta de reflexión:** ¿Por qué crees que la GRU es más rápida de entrenar que la LSTM?